In [64]:
import pandas as pd
import numpy as np

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score


In [2]:
df = pd.read_csv("C:/Users/slmkc/Downloads/PhaseI/Churn_model/data/raw/cell2celltrain.csv")


In [3]:
df

,CustomerID,Churn,MonthlyRevenue,MonthlyMinutes,TotalRecurringCharge,DirectorAssistedCalls,OverageMinutes,RoamingCalls,PercChangeMinutes,PercChangeRevenues,...,ReferralsMadeBySubscriber,IncomeGroup,OwnsMotorcycle,AdjustmentsToCreditRating,HandsetPrice,MadeCallToRetentionTeam,CreditRating,PrizmCode,Occupation,MaritalStatus
0,3000002,Yes,24.00,219.0,22.0,0.25,0.0,0.0,-157.0,-19.0,...,0,4,No,0,30,Yes,1-Highest,Suburban,Professional,No
1,3000010,Yes,16.99,10.0,17.0,0.00,0.0,0.0,-4.0,0.0,...,0,5,No,0,30,No,4-Medium,Suburban,Professional,Yes
2,3000014,No,38.00,8.0,38.0,0.00,0.0,0.0,-2.0,0.0,...,0,6,No,0,Unknown,No,3-Good,Town,Crafts,Yes
3,3000022,No,82.28,1312.0,75.0,1.24,0.0,0.0,157.0,8.1,...,0,6,No,0,10,No,4-Medium,Other,Other,No
4,3000026,Yes,17.14,0.0,17.0,0.00,0.0,0.0,0.0,-0.2,...,0,9,No,1,10,No,1-Highest,Other,Professional,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51042,3399958,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,6,No,0,60,No,1-Highest,Suburban,Other,Yes
51043,3399974,No,95.17,1745.0,85.0,0.99,45.0,4.7,122.0,15.9,...,0,9,No,1,60,No,3-Good,Other,Other,No
51044,3399978,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,7,No,1,80,No,5-Low,Other,Clerical,No
51045,3399990,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,9,No,0,30,No,5-Low,Other,Other,No


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51047 entries, 0 to 51046
Data columns (total 58 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CustomerID                 51047 non-null  int64  
 1   Churn                      51047 non-null  object 
 2   MonthlyRevenue             50891 non-null  float64
 3   MonthlyMinutes             50891 non-null  float64
 4   TotalRecurringCharge       50891 non-null  float64
 5   DirectorAssistedCalls      50891 non-null  float64
 6   OverageMinutes             50891 non-null  float64
 7   RoamingCalls               50891 non-null  float64
 8   PercChangeMinutes          50680 non-null  float64
 9   PercChangeRevenues         50680 non-null  float64
 10  DroppedCalls               51047 non-null  float64
 11  BlockedCalls               51047 non-null  float64
 12  UnansweredCalls            51047 non-null  float64
 13  CustomerCareCalls          51047 non-null  flo

In [5]:
df.shape

(51047, 58)

In [6]:
print("Churn Value", df["Churn"].value_counts().to_dict())

Churn Value {'No': 36336, 'Yes': 14711}


In [7]:
print("Churn rate:", round((df["Churn"] == "Yes").mean(), 4))

Churn rate: 0.2882


In [8]:
print("CustomerID unique", df["CustomerID"].is_unique)

CustomerID unique True


In [9]:
print("Duplicate rows ignoring ID", df.drop(columns = "CustomerID").duplicated().sum())

Duplicate rows ignoring ID 0


In [10]:
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending = False)

In [11]:
print(missing)

AgeHH1                   909
AgeHH2                   909
PercChangeMinutes        367
PercChangeRevenues       367
MonthlyRevenue           156
MonthlyMinutes           156
TotalRecurringCharge     156
DirectorAssistedCalls    156
OverageMinutes           156
RoamingCalls             156
ServiceArea               24
Handsets                   1
HandsetModels              1
CurrentEquipmentDays       1
dtype: int64


In [12]:
print("Rows with atleast one missing value", df.isnull().any(axis = 1).sum())

Rows with atleast one missing value 1295


In [13]:
print("AgeHH1 equal to 0", (df["AgeHH1"] == 0).sum())

AgeHH1 equal to 0 13917


In [14]:
numeric_cols = df.select_dtypes(include = "number").columns

text_cols = df.select_dtypes(exclude = "number").columns

In [15]:
print("Numeric columns", len(numeric_cols))

Numeric columns 35


In [16]:
print("Text columns", len(text_cols))

Text columns 23


In [17]:
for col in text_cols:
    print(col, " -- ", df[col].nunique(), "categories")

Churn  --  2 categories
ServiceArea  --  747 categories
ChildrenInHH  --  2 categories
HandsetRefurbished  --  2 categories
HandsetWebCapable  --  2 categories
TruckOwner  --  2 categories
RVOwner  --  2 categories
Homeownership  --  2 categories
BuysViaMailOrder  --  2 categories
RespondsToMailOffers  --  2 categories
OptOutMailings  --  2 categories
NonUSTravel  --  2 categories
OwnsComputer  --  2 categories
HasCreditCard  --  2 categories
NewCellphoneUser  --  2 categories
NotNewCellphoneUser  --  2 categories
OwnsMotorcycle  --  2 categories
HandsetPrice  --  16 categories
MadeCallToRetentionTeam  --  2 categories
CreditRating  --  7 categories
PrizmCode  --  4 categories
Occupation  --  8 categories
MaritalStatus  --  3 categories


In [18]:
y = (df["Churn"] == "Yes").astype(int)

In [19]:
results = {}

for col in df.select_dtypes(include = "number").columns.drop("CustomerID"):
    x = df[col].fillna(df[col].median())
    if x.nunique() > 1:
        auc = roc_auc_score(y, x)
        results[col] = round(max(auc, 1 - auc), 3)

In [20]:
print(pd.Series(results).sort_values(ascending = False).head(8))

CurrentEquipmentDays    0.574
TotalRecurringCharge    0.542
MonthlyMinutes          0.540
OffPeakCallsInOut       0.535
InboundCalls            0.535
PeakCallsInOut          0.531
PercChangeMinutes       0.531
CustomerCareCalls       0.530
dtype: float64


In [21]:
print((y.groupby(df["MadeCallToRetentionTeam"]).mean() * 100).round(3))

MadeCallToRetentionTeam
No     28.244
Yes    45.043
Name: Churn, dtype: float64


In [22]:
print(df["MadeCallToRetentionTeam"].value_counts())

MadeCallToRetentionTeam
No     49302
Yes     1745
Name: count, dtype: int64


In [23]:
print((y.groupby(df["RetentionCalls"]).mean() * 100).round(3))

RetentionCalls
0     28.244
1     44.997
2     44.167
3     50.000
4    100.000
Name: Churn, dtype: float64


In [24]:
drop_cols = ["CustomerID", "ServiceArea"]

retention_cols = ["MadeCallToRetentionTeam", "RetentionCalls", "RetentionOffersAccepted"]


In [27]:
X_all = df.drop(columns = ["Churn"] + drop_cols)

#handling age 0 case

for col in ["AgeHH1", "AgeHH2"]:
    X_all[col] = X_all[col].replace(0, np.nan)

In [28]:
print(X_all.shape)

(51047, 55)


In [29]:
print("AgeHH1 missing now:", X_all["AgeHH1"].isnull().sum())

AgeHH1 missing now: 14826


In [32]:
X_train_all, X_test_all, y_train, y_test = train_test_split(X_all, y, test_size = 0.2, stratify = y, random_state = 42)

X_train = X_train_all.drop(columns = retention_cols)

X_test = X_test_all.drop(columns = retention_cols)

In [39]:
print("Train", X_train.shape, "Test", X_test.shape)

Train (40837, 52) Test (10210, 52)


In [34]:
print("Churn rate train", round(y_train.mean(), 4), "| test", round(y_test.mean(), 4))

Churn rate train 0.2882 | test 0.2881


In [35]:
print("Churn inside X_train", "Churn" in X_train.columns)

Churn inside X_train False


In [36]:
print("CustomerID inside X_train", "CustomerID" in X_train.columns)

CustomerID inside X_train False


In [38]:
print("Customers in both train and test", len(set(df.loc[X_train.index, "CustomerID"]) & set(df.loc[X_test.index, "CustomerID"])))

Customers in both train and test 0


In [58]:
def build_preprocess(X):
    num_cols = X.select_dtypes(include = "number").columns.tolist()
    cat_cols = X.select_dtypes(exclude = "number").columns.tolist()

    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy = "median", add_indicator = True), StandardScaler()), num_cols),
        ("cat", OneHotEncoder(handle_unknown = "ignore", sparse_output = False), cat_cols),
    ])

In [59]:
def build_models(X):
    return {
        "Logistic Regression": make_pipeline(build_preprocess(X), LogisticRegression(max_iter = 2000)),
        "Gradient Boosting": make_pipeline(build_preprocess(X), HistGradientBoostingClassifier(learning_rate = 0.05, max_iter = 200, random_state = 42)),
    }

In [60]:
print("Numeric", len(X_train.select_dtypes(include = "number").columns), "| Categorical", len(X_train.select_dtypes(exclude = "number").columns))

Numeric 32 | Categorical 20


In [61]:
pre = build_preprocess(X_train)
pre.fit(X_train)
Xt = pre.transform(X_train)

In [62]:
print("Columns before", X_train.shape[1], "| after", Xt.shape[1])


Columns before 52 | after 113


In [63]:
print("Missing Value after Preprocessing", int(np.isnan(Xt).sum()))

Missing Value after Preprocessing 0


In [65]:
cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)

In [68]:
rows = []

for name, model in build_models(X_train).items():
    s = cross_validate(model, X_train, y_train, cv = cv, scoring = ["roc_auc", "average_precision"])
    rows.append({"model": name,
                 "roc_auc": s["test_roc_auc"].mean(),
                 "roc_auc_std": s["test_roc_auc"].std(),
                 "pr_auc": s["test_average_precision"].mean()})
    print(name)

Logistic Regression
Gradient Boosting


In [69]:
print(pd.DataFrame(rows).round(4).to_string(index = False))


              model  roc_auc  roc_auc_std  pr_auc
Logistic Regression   0.6161       0.0099  0.3817
  Gradient Boosting   0.6760       0.0103  0.4538


In [70]:
print("Random guessing: ROC-AUC 0.5 | PR-AUC about", round(y_train.mean(), 3))

Random guessing: ROC-AUC 0.5 | PR-AUC about 0.288


In [73]:
rows = []

for label, X_data in [("without retention columns", X_train), ("with retention columns", X_train_all)]:
    for name, model in build_models(X_data).items():
        s = cross_val_score(model, X_data, y_train, cv = cv, scoring = "roc_auc")
        rows.append({"columns": label,
                     "model": name,
                     "roc_auc": round(s.mean(), 4),
                     "std": round(s.std(), 4)})
        print(label, "|" , name)

without retention columns | Logistic Regression
without retention columns | Gradient Boosting
with retention columns | Logistic Regression
with retention columns | Gradient Boosting


In [74]:
print(pd.DataFrame(rows).to_string(index = False))

                  columns               model  roc_auc    std
without retention columns Logistic Regression   0.6161 0.0099
without retention columns   Gradient Boosting   0.6760 0.0103
   with retention columns Logistic Regression   0.6210 0.0100
   with retention columns   Gradient Boosting   0.6782 0.0086
